In [1]:
# ============================================================================
# NOTEBOOK 03 — PROJECT ROOT INITIALIZATION
# ============================================================================
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root. Current working directory: "
        f"{current}"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
METADATA_DIR = DATA_DIR / "metadata"
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
MERGED_DIR = DATA_DIR / "merged"

print("=" * 70)
print("PROJECT ROOT")
print("=" * 70)
print(PROJECT_ROOT)


PROJECT ROOT
C:\Users\lab\Minor_project\Parkinson_gait


# Notebook 03 — Linear + Angular Kinematic Data Merging

## Research Purpose & Pipeline Handoff
This notebook implements the deterministic multimodal data integration stage for **Parkinson_Gait_CIGTNet** (*Clinically-Informed Multiscale Graph-Transformer Framework for Explainable Parkinsonian Gait Severity Assessment from Multisensor Time-Series Data*).

Building directly upon the authoritative trial pairing index validated in **Notebook 02**, this notebook merges corresponding raw linear and angular kinematic time-series streams for each walking trial into a single unified multimodal representation while preserving:
1. **Temporal Integrity**: Exact timebase alignment, common temporal intervals without arbitrary truncation, and documented sampling properties.
2. **Feature Identity & Provenance**: Collision-free namespace (`lin__` and `ang__` prefixes) with complete feature provenance tracking (`merged_feature_provenance.csv`).
3. **Signal Value Fidelity**: Zero alteration of raw values (no smoothing, no scaling, no normalization, no filtering, no imputation).
4. **Data Isolation**: Complete read-only protection of `data/raw/`.

### Downstream Contract for Notebook 04
Notebook 04 will consume the merged kinematics directory (`data/interim/merged_kinematics/`) and the manifest (`data/metadata/merged_trial_manifest.csv`) without needing to rediscover modality relationships, trial keys, or feature identities.


In [2]:
# ============================================================================
# NOTEBOOK 03 — STABLE ENVIRONMENT CONFIGURATION
# ============================================================================
import os
import sys
import time
import glob
import json
import hashlib
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import xlrd
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
METADATA_DIR = DATA_DIR / "metadata"
INTERIM_DIR = DATA_DIR / "interim"
# Authoritative primary output directory as specified in Part 2 prompt
MERGED_DIR = DATA_DIR / "merged"
# Backward-compatibility mirror directory
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = PROJECT_ROOT / "figures" / "preprocessing"
for d in [METADATA_DIR, MERGED_DIR, TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# Methodological tolerances:
# TIME_ABS_TOL: numerical tolerance for matching timestamps (0.1 ms)
# GRID_DT_REL_TOL & GRID_DT_ABS_TOL: accounts for millisecond quantization (0.006s vs 0.007s)
# inherent to 3-decimal export of 150 Hz sampling (1/150 = 0.00667s)
TIME_ABS_TOL = 1e-4
GRID_DT_REL_TOL = 0.20
GRID_DT_ABS_TOL = 1.5e-3
print("=" * 70)
print("NOTEBOOK 03 ENVIRONMENT")
print("=" * 70)
print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {platform.python_version()}")
print(f"Pandas       : {pd.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"xlrd         : {xlrd.__version__}")
print(f"Primary output: {MERGED_DIR}")
print("=" * 70)


NOTEBOOK 03 ENVIRONMENT
Project root : C:\Users\lab\Minor_project\Parkinson_gait
Python       : 3.11.0
Pandas       : 3.0.6
NumPy        : 2.4.6
xlrd         : 2.0.2
Primary output: C:\Users\lab\Minor_project\Parkinson_gait\data\merged


## 1. Pre-Execution Raw Data Integrity Snapshot

To guarantee that the raw dataset in `data/raw/` remains strictly read-only and immutable throughout the merging pipeline, we record a cryptographic/size manifest of all files prior to any processing.


In [3]:
# ============================================================================
# 1. PRE-EXECUTION RAW DATA INTEGRITY SNAPSHOT
# ============================================================================

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

raw_files_pre = []
for root, _, files in os.walk(RAW_DIR):
    for name in sorted(files):
        p = Path(root) / name
        raw_files_pre.append({
            "relative_path": str(p.relative_to(RAW_DIR)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

df_raw_pre = (
    pd.DataFrame(raw_files_pre)
    .sort_values("relative_path")
    .reset_index(drop=True)
)

RAW_INTEGRITY_PRE_PATH = TABLES_DIR / "raw_integrity_pre.csv"
df_raw_pre.to_csv(RAW_INTEGRITY_PRE_PATH, index=False)

print("=" * 70)
print("RAW DATA PRE-EXECUTION INTEGRITY")
print("=" * 70)
print(f"Files recorded : {len(df_raw_pre)}")
print(f"Total bytes    : {int(df_raw_pre['size_bytes'].sum()):,}")
print(f"Manifest       : {RAW_INTEGRITY_PRE_PATH.relative_to(PROJECT_ROOT)}")
print("=" * 70)

if df_raw_pre.empty:
    raise RuntimeError("No raw files were discovered under data/raw/.")

RAW DATA PRE-EXECUTION INTEGRITY
Files recorded : 4831
Total bytes    : 2,700,108,888
Manifest       : results\tables\raw_integrity_pre.csv


## 2. Input Validation (Notebook 02 Trial Metadata Index)

We consume the authoritative index produced by Notebook 02 (`data/metadata/trial_metadata_index.csv`). The schema and referenced files are validated to ensure only confirmed `PAIRED` trials are processed.


In [4]:
# ============================================================================
# 2. NOTEBOOK 02 HANDOFF VALIDATION
# ============================================================================

TRIAL_INDEX_PATH = METADATA_DIR / "trial_metadata_index.csv"

if not TRIAL_INDEX_PATH.exists():
    raise FileNotFoundError(
        f"Required Notebook 02 artifact not found: {TRIAL_INDEX_PATH}"
    )

trial_index_df = pd.read_csv(TRIAL_INDEX_PATH)

required_cols = [
    "trial_key", "subject_id", "condition", "trial_id",
    "linear_path", "angular_path", "pairing_status"
]
missing_cols = [c for c in required_cols if c not in trial_index_df.columns]

if missing_cols:
    raise RuntimeError(
        f"Notebook 02 index is missing required columns: {missing_cols}"
    )

if trial_index_df["trial_key"].duplicated().any():
    raise RuntimeError("Notebook 02 trial_key is not unique.")

paired_trials_df = trial_index_df[
    trial_index_df["pairing_status"].astype(str).str.upper().eq("PAIRED")
].copy().reset_index(drop=True)

if paired_trials_df.empty:
    raise RuntimeError("Notebook 02 contains no PAIRED trials.")

def resolve_raw_path(value):
    p = Path(str(value))
    return p if p.is_absolute() else RAW_DIR / p

paired_trials_df["linear_full_path"] = paired_trials_df["linear_path"].map(resolve_raw_path)
paired_trials_df["angular_full_path"] = paired_trials_df["angular_path"].map(resolve_raw_path)

paired_trials_df["linear_exists"] = paired_trials_df["linear_full_path"].map(Path.exists)
paired_trials_df["angular_exists"] = paired_trials_df["angular_full_path"].map(Path.exists)

if not paired_trials_df["linear_exists"].all() or not paired_trials_df["angular_exists"].all():
    problems = paired_trials_df[
        (~paired_trials_df["linear_exists"]) |
        (~paired_trials_df["angular_exists"])
    ]
    problems.to_csv(TABLES_DIR / "merge_input_file_problems.csv", index=False)
    raise RuntimeError(
        "Notebook 02 paired trials reference missing files. "
        "See results/tables/merge_input_file_problems.csv."
    )

print("=" * 70)
print("NOTEBOOK 02 HANDOFF VALIDATION")
print("=" * 70)
print(f"Rows in index     : {len(trial_index_df)}")
print(f"PAIRED trials     : {len(paired_trials_df)}")
print("Linear files      : all present")
print("Angular files     : all present")
print("=" * 70)

NOTEBOOK 02 HANDOFF VALIDATION
Rows in index     : 884
PAIRED trials     : 884
Linear files      : all present
Angular files     : all present


## 3. Kinematic File Parser & Temporal Coordinate Discovery

As established during dataset discovery, the kinematic files in `C3Dfiles` have a `.csv` naming extension but are structured as **OLE2 Excel binary spreadsheets** (BIFF8 format). 

Each file contains:
- **Rows 0–3**: Gait event marker tables (e.g., heel-strike and toe-off frame indices).
- **Row 4**: Channel header names.
- **Row 5**: Measurement engineering units.
- **Rows 6+**: Discrete time-series measurement rows.

We implement an efficient parser using `xlrd` that extracts column headers, engineering units, and numerical arrays while identifying the temporal coordinates.


In [5]:
# ============================================================================
# 3. OLE2/BIFF8 KINEMATIC PARSER + TEMPORAL DISCOVERY
# ============================================================================

def make_unique_column_names(names):
    """
    Make source column names unique without removing, merging, or reordering
    any source columns.

    Example:
        ML, ML, Vertical, ML
    becomes:
        ML, ML.1, Vertical, ML.2

    The first occurrence keeps the original name.
    """

    counts = {}
    unique_names = []

    for raw_name in names:
        name = str(raw_name).strip()

        if name not in counts:
            counts[name] = 0
            unique_names.append(name)
        else:
            counts[name] += 1
            unique_names.append(f"{name}.{counts[name]}")

    return unique_names


def read_kinematics_xls(filepath):
    """
    Parse an OLE2/BIFF8 kinematic spreadsheet.

    Important:
    - Source columns are NOT discarded.
    - Duplicate source names are deterministically disambiguated.
    - Source column order is preserved.
    - Source units remain attached to their original column positions.
    - Raw numeric values are not transformed.
    """

    filepath = Path(filepath)

    wb = xlrd.open_workbook(
        str(filepath),
        on_demand=True
    )

    try:
        sheet = wb.sheet_by_index(0)

        if sheet.nrows < 7:
            raise ValueError(
                f"Insufficient rows in kinematic file: {filepath}"
            )

        # ------------------------------------------------------------------
        # Source header rows
        # ------------------------------------------------------------------
        raw_columns = [
            str(sheet.cell_value(4, c)).strip()
            for c in range(sheet.ncols)
        ]

        units = [
            str(sheet.cell_value(5, c)).strip()
            for c in range(sheet.ncols)
        ]

        if any(name == "" for name in raw_columns):
            blank_positions = [
                i for i, name in enumerate(raw_columns)
                if name == ""
            ]

            raise ValueError(
                f"Blank source column name(s) found in {filepath}. "
                f"Column positions: {blank_positions[:10]}"
            )

        # ------------------------------------------------------------------
        # IMPORTANT:
        # Duplicate names are valid in the source file.
        # Preserve every column and make names unique only for DataFrame use.
        # ------------------------------------------------------------------
        duplicate_mask = (
            pd.Series(raw_columns)
            .duplicated(keep=False)
        )

        duplicate_names = sorted(
            set(
                pd.Series(raw_columns)[duplicate_mask]
                .tolist()
            )
        )

        columns = make_unique_column_names(raw_columns)

        if len(columns) != len(set(columns)):
            raise RuntimeError(
                f"Internal column-name disambiguation failed: {filepath}"
            )

        # ------------------------------------------------------------------
        # Read numerical data rows.
        # Rows 0-3 : event information
        # Row 4    : channel names
        # Row 5    : units
        # Row 6+   : time-series data
        # ------------------------------------------------------------------
        n_rows = sheet.nrows - 6

        matrix = np.full(
            (n_rows, sheet.ncols),
            np.nan,
            dtype=float
        )

        for c in range(sheet.ncols):

            values = sheet.col_values(
                c,
                start_rowx=6
            )

            for r, value in enumerate(values):

                if value in ("", None):
                    continue

                try:
                    matrix[r, c] = float(value)

                except (TypeError, ValueError):

                    # Preserve the row/column position while representing
                    # non-numeric cells as missing numerical observations.
                    matrix[r, c] = np.nan

        df = pd.DataFrame(
            matrix,
            columns=columns
        )

        return columns, units, df

    finally:
        wb.release_resources()


def find_column(columns, target):
    """
    Find a unique temporal coordinate column.

    Exact name matching is used after stripping whitespace and ignoring case.
    """

    matches = [
        c
        for c in columns
        if str(c).strip().lower() == target.lower()
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one {target} column; "
            f"found {matches}"
        )

    return matches[0]


# ============================================================================
# PARSER DISCOVERY TEST
# ============================================================================

sample = paired_trials_df.iloc[0]

sample_lin_cols, sample_lin_units, sample_lin_df = (
    read_kinematics_xls(
        sample["linear_full_path"]
    )
)

sample_ang_cols, sample_ang_units, sample_ang_df = (
    read_kinematics_xls(
        sample["angular_full_path"]
    )
)

# ---------------------------------------------------------------------------
# Temporal coordinates
# ---------------------------------------------------------------------------

lin_time_col = find_column(
    sample_lin_cols,
    "Time"
)

lin_frame_col = find_column(
    sample_lin_cols,
    "Frame"
)

ang_time_col = find_column(
    sample_ang_cols,
    "Time"
)

ang_frame_col = find_column(
    sample_ang_cols,
    "Frame"
)


# ============================================================================
# DUPLICATE-SOURCE-COLUMN REPORT
# ============================================================================

def duplicate_base_names(columns):
    """
    Return original base names that occurred more than once.
    """

    counts = pd.Series(columns).value_counts()

    return counts[counts > 1].sort_index()


linear_duplicate_names = duplicate_base_names(
    [
        str(x).strip()
        for x in xlrd.open_workbook(
            str(sample["linear_full_path"]),
            on_demand=True
        ).sheet_by_index(0).row_values(4)
    ]
)

angular_duplicate_names = duplicate_base_names(
    [
        str(x).strip()
        for x in xlrd.open_workbook(
            str(sample["angular_full_path"]),
            on_demand=True
        ).sheet_by_index(0).row_values(4)
    ]
)


# ============================================================================
# REPORT
# ============================================================================

print("=" * 70)
print("PARSER / TEMPORAL DISCOVERY")
print("=" * 70)

print(f"Linear shape   : {sample_lin_df.shape}")
print(f"Angular shape  : {sample_ang_df.shape}")

print(f"Linear time    : {lin_time_col}")
print(f"Linear frame   : {lin_frame_col}")
print(f"Angular time   : {ang_time_col}")
print(f"Angular frame  : {ang_frame_col}")

print("-" * 70)

if len(linear_duplicate_names):
    print("Linear duplicate source names preserved:")
    print(linear_duplicate_names.to_dict())
else:
    print("Linear duplicate source names: none")

if len(angular_duplicate_names):
    print("Angular duplicate source names preserved:")
    print(angular_duplicate_names.to_dict())
else:
    print("Angular duplicate source names: none")

print("-" * 70)

print(
    f"Unique linear columns  : "
    f"{len(sample_lin_cols)}"
)

print(
    f"Unique angular columns : "
    f"{len(sample_ang_cols)}"
)

print("=" * 70)

PARSER / TEMPORAL DISCOVERY
Linear shape   : (671, 137)
Angular shape  : (671, 47)
Linear time    : Time
Linear frame   : Frame
Angular time   : Time
Angular frame  : Frame
----------------------------------------------------------------------
Linear duplicate source names preserved:
{'ML': 45, 'Vertical': 45}
Angular duplicate source names preserved:
{'Ankle Add/Abduction': 2, 'Ankle Dorsi/Plantarflexion': 2, 'Elbow Flx/Extension': 2, 'Elbow Pron/Supination': 2, 'Foot DF/Plantarflexion': 2, 'Foot Int/External Rotation': 2, 'Hip Flexion/Extension': 2, 'Hip Int/External Rotation': 2, 'Knee Flx/Extension': 2, 'Knee Int/External Rotation': 2, 'Pelvic Rotation': 2, 'Pelvic Tilt': 2, 'Shoulder Flx/Extension': 2, 'Shoulder Int/External Rotation': 2}
----------------------------------------------------------------------
Unique linear columns  : 137
Unique angular columns : 47


## 4. Feature Provenance & Column Collision Prevention

To prevent column collisions when linear and angular streams share identical coordinate or anatomical names, we enforce a strict prefixing convention:
- `lin__<feature_name>` for all linear kinematic features.
- `ang__<feature_name>` for all angular kinematic features.
- Dedicated temporal coordinates `frame` and `time` are placed at the beginning of the merged stream.

Every feature's origin, units, modality, and original position are documented in `data/metadata/merged_feature_provenance.csv` to support explainability and biomechanical graph construction in later notebooks.


In [6]:
# ============================================================================
# 4. CANONICAL MERGED SCHEMA + FEATURE PROVENANCE
# ============================================================================

def signal_cols(columns, time_col, frame_col):
    return [c for c in columns if c not in {time_col, frame_col}]

canonical_linear_cols = signal_cols(sample_lin_cols, lin_time_col, lin_frame_col)
canonical_angular_cols = signal_cols(sample_ang_cols, ang_time_col, ang_frame_col)

canonical_linear_units = {
    c: sample_lin_units[sample_lin_cols.index(c)]
    for c in canonical_linear_cols
}
canonical_angular_units = {
    c: sample_ang_units[sample_ang_cols.index(c)]
    for c in canonical_angular_cols
}

merged_columns = (
    ["frame", "time"]
    + [f"lin__{c}" for c in canonical_linear_cols]
    + [f"ang__{c}" for c in canonical_angular_cols]
)

provenance_records = [
    {
        "merged_feature_name": "frame",
        "source_modality": "shared_temporal",
        "source_feature_name": lin_frame_col,
        "unit_if_available": sample_lin_units[sample_lin_cols.index(lin_frame_col)],
        "source_column_position": sample_lin_cols.index(lin_frame_col),
    },
    {
        "merged_feature_name": "time",
        "source_modality": "shared_temporal",
        "source_feature_name": lin_time_col,
        "unit_if_available": sample_lin_units[sample_lin_cols.index(lin_time_col)],
        "source_column_position": sample_lin_cols.index(lin_time_col),
    },
]

for c in canonical_linear_cols:
    provenance_records.append({
        "merged_feature_name": f"lin__{c}",
        "source_modality": "linear_kinematics",
        "source_feature_name": c,
        "unit_if_available": canonical_linear_units[c],
        "source_column_position": sample_lin_cols.index(c),
    })

for c in canonical_angular_cols:
    provenance_records.append({
        "merged_feature_name": f"ang__{c}",
        "source_modality": "angular_kinematics",
        "source_feature_name": c,
        "unit_if_available": canonical_angular_units[c],
        "source_column_position": sample_ang_cols.index(c),
    })

provenance_df = pd.DataFrame(provenance_records)
PROVENANCE_PATH = METADATA_DIR / "merged_feature_provenance.csv"
provenance_df.to_csv(PROVENANCE_PATH, index=False)

if provenance_df["merged_feature_name"].tolist() != merged_columns:
    raise RuntimeError("Provenance order does not equal merged schema order.")

print("=" * 70)
print("CANONICAL FEATURE SCHEMA")
print("=" * 70)
print(f"Linear signal features  : {len(canonical_linear_cols)}")
print(f"Angular signal features : {len(canonical_angular_cols)}")
print(f"Total merged columns    : {len(merged_columns)}")
print(f"Provenance              : {PROVENANCE_PATH.relative_to(PROJECT_ROOT)}")
print("=" * 70)

CANONICAL FEATURE SCHEMA
Linear signal features  : 135
Angular signal features : 45
Total merged columns    : 182
Provenance              : data\metadata\merged_feature_provenance.csv


## 5. Temporal Alignment Engine & Quality Control

We follow a strict alignment hierarchy to merge paired streams without silent data alteration:
1. **Case 1 (`EXACTLY_ALIGNED`)**: When sequence lengths and timestamps match sample-by-sample across the entire duration, merge directly.
2. **Case 2 (`SAME_GRID_DIFFERENT_LENGTH`)**: When sampling intervals and initial timestamps match but endpoints differ, compute the common valid temporal interval $[\max(t_{\text{start}}), \min(t_{\text{end}})]$. Discarded samples are recorded and reported in the manifest.
3. **Case 3 (`INCOMPATIBLE_TIMEBASE`)**: Flag for review if sampling intervals differ.
4. **Case 4 (`MISSING_TIME_INFORMATION`)**: Flag for review if time coordinates are missing.

### Quality Control Assertions Per Trial:
- Merged row count $> 0$.
- Monotonic timestamps with zero duplicate timestamps.
- Feature count conservation: $\text{merged\_features} = \text{linear\_features} + \text{angular\_features}$.
- Raw value fidelity: exact value preservation against source arrays.


In [7]:
# ============================================================================
# 5. TEMPORAL ALIGNMENT ENGINE
# ============================================================================

def validate_time_frame(df, time_col, frame_col):
    time = pd.to_numeric(df[time_col], errors="coerce").to_numpy(float)
    frame = pd.to_numeric(df[frame_col], errors="coerce").to_numpy(float)

    if len(time) == 0:
        raise ValueError("MISSING_TIME_INFORMATION: empty sequence.")

    if not np.isfinite(time).all() or not np.isfinite(frame).all():
        raise ValueError("MISSING_TIME_INFORMATION: Time/Frame contains non-finite values.")

    if len(time) > 1 and not np.all(np.diff(time) > 0):
        raise ValueError("NON_MONOTONIC_TIME: Time is not strictly increasing.")

    if len(frame) > 1 and not np.all(np.diff(frame) > 0):
        raise ValueError("NON_MONOTONIC_FRAME: Frame is not strictly increasing.")

    return time, frame

def median_dt(time):
    return float(np.median(np.diff(time))) if len(time) > 1 else np.nan

def uniform_grid(time):
    if len(time) < 3:
        return True
    diffs = np.diff(time)
    return bool(np.allclose(diffs, np.median(diffs), rtol=GRID_DT_REL_TOL, atol=GRID_DT_ABS_TOL))

def schema_matches(cols, units, expected_cols, expected_units, time_col, frame_col):
    if time_col not in cols or frame_col not in cols:
        return False, "required Time/Frame column missing"

    observed = signal_cols(cols, time_col, frame_col)
    if observed != expected_cols:
        return False, "signal feature names/order differ from canonical schema"

    for c in expected_cols:
        observed_unit = units[cols.index(c)]
        expected_unit = expected_units.get(c, "")
        if observed_unit and expected_unit and observed_unit != expected_unit:
            return False, f"unit mismatch for {c}: {observed_unit} != {expected_unit}"

    return True, ""

def existing_sample_matches(t1, f1, t2, f2, tol=TIME_ABS_TOL):
    i = j = 0
    pairs = []
    while i < len(t1) and j < len(t2):
        dt = t1[i] - t2[j]
        if abs(dt) <= tol and f1[i] == f2[j]:
            pairs.append((i, j))
            i += 1
            j += 1
        elif t1[i] < t2[j]:
            i += 1
        else:
            j += 1
    return pairs

def merge_trial(trial_row):
    try:
        lin_cols, lin_units, lin_df = read_kinematics_xls(trial_row["linear_full_path"])
        ang_cols, ang_units, ang_df = read_kinematics_xls(trial_row["angular_full_path"])

        ltc, lfc = find_column(lin_cols, "Time"), find_column(lin_cols, "Frame")
        atc, afc = find_column(ang_cols, "Time"), find_column(ang_cols, "Frame")

        ok, reason = schema_matches(lin_cols, lin_units, canonical_linear_cols, canonical_linear_units, ltc, lfc)
        if not ok:
            raise ValueError(f"SCHEMA_MISMATCH: linear: {reason}")

        ok, reason = schema_matches(ang_cols, ang_units, canonical_angular_cols, canonical_angular_units, atc, afc)
        if not ok:
            raise ValueError(f"SCHEMA_MISMATCH: angular: {reason}")

        lt, lf = validate_time_frame(lin_df, ltc, lfc)
        at, af = validate_time_frame(ang_df, atc, afc)

        ldt, adt = median_dt(lt), median_dt(at)

        if not uniform_grid(lt) or not uniform_grid(at):
            raise ValueError("NON_UNIFORM_GRID: one or both streams are non-uniform.")

        if not np.isclose(ldt, adt, rtol=GRID_DT_REL_TOL, atol=GRID_DT_ABS_TOL):
            raise ValueError("SAMPLING_INTERVAL_MISMATCH: sampling intervals differ.")

        if (
            len(lt) == len(at)
            and np.array_equal(lf, af)
            and np.allclose(lt, at, rtol=0.0, atol=TIME_ABS_TOL)
        ):
            li = np.arange(len(lt))
            ai = np.arange(len(at))
            alignment_status = "EXACTLY_ALIGNED"
            alignment_method = "sample_by_sample_frame_and_time"
        else:
            pairs = existing_sample_matches(lt, lf, at, af)
            if not pairs:
                raise ValueError("NO_COMMON_SAMPLES: no common existing samples.")

            if len(pairs) != min(len(lt), len(at)):
                raise ValueError("INCOMPATIBLE_TIMEBASE: streams do not share complete sample grid of shorter sequence.")

            li = np.array([p[0] for p in pairs], dtype=int)
            ai = np.array([p[1] for p in pairs], dtype=int)
            alignment_status = "SAME_GRID_DIFFERENT_LENGTH"
            alignment_method = "existing_sample_grid_intersection"

        if not np.array_equal(lf[li], af[ai]):
            raise ValueError("INCOMPATIBLE_TIMEBASE: retained frames disagree.")

        if not np.allclose(lt[li], at[ai], rtol=0.0, atol=TIME_ABS_TOL):
            raise ValueError("INCOMPATIBLE_TIMEBASE: retained times disagree.")

        # Dictionary construction to avoid DataFrame fragmentation
        merged_data = {
            "frame": lf[li],
            "time": lt[li],
        }
        for c in canonical_linear_cols:
            merged_data[f"lin__{c}"] = lin_df[c].to_numpy()[li]
        for c in canonical_angular_cols:
            merged_data[f"ang__{c}"] = ang_df[c].to_numpy()[ai]

        merged = pd.DataFrame(merged_data)

        if merged.columns.tolist() != merged_columns:
            raise RuntimeError("Internal merged-schema ordering error.")

        record = {
            "trial_key": str(trial_row["trial_key"]),
            "subject_id": trial_row["subject_id"],
            "condition": trial_row["condition"],
            "trial_id": trial_row["trial_id"],
            "linear_file": str(trial_row["linear_path"]),
            "angular_file": str(trial_row["angular_path"]),
            "pairing_status": trial_row["pairing_status"],
            "merge_status": "SUCCESS",
            "failure_reason": "",
            "alignment_status": alignment_status,
            "alignment_method": alignment_method,
            "linear_original_length": len(lin_df),
            "angular_original_length": len(ang_df),
            "merged_length": len(merged),
            "linear_start_time": float(lt[0]),
            "linear_end_time": float(lt[-1]),
            "angular_start_time": float(at[0]),
            "angular_end_time": float(at[-1]),
            "merged_start_time": float(merged["time"].iloc[0]),
            "merged_end_time": float(merged["time"].iloc[-1]),
            "linear_sampling_interval": ldt,
            "angular_sampling_interval": adt,
            "discarded_linear_samples": len(lin_df) - len(li),
            "discarded_angular_samples": len(ang_df) - len(ai),
            "output_path": str(Path("data") / "merged" / f"{trial_row['trial_key']}.csv"),
            "validation_status": "PENDING"
        }

        return record, merged, None

    except Exception as exc:
        failure = {
            "trial_key": str(trial_row["trial_key"]),
            "subject_id": trial_row["subject_id"],
            "condition": trial_row["condition"],
            "trial_id": trial_row["trial_id"],
            "failure_stage": "merge_trial",
            "failure_reason": str(exc),
            "linear_file": str(trial_row["linear_path"]),
            "angular_file": str(trial_row["angular_path"]),
        }
        return None, None, failure



## 6. Batch Execution Across All Valid Paired Trials

We execute the merging engine across all paired trials. Each merged trial is written as a standardized CSV file into `data/interim/merged_kinematics/`. The full audit record is captured for every trial.


## Reproducibility safeguard — stale-output cleanup

Before each batch execution, Notebook 03 removes only previously generated `*_merged.csv` files from `data/interim/merged_kinematics/`. This prevents a trial that fails during the current run from being incorrectly represented by a stale merged file from an earlier execution.

No files under `data/raw/` are modified.


In [8]:
# ============================================================================
# 6. FRESH-RUN CLEANUP + BATCH EXECUTION
# ============================================================================

for p in sorted(MERGED_DIR.glob("*.csv")):
    p.unlink()

manifest_records = []
failure_records = []

start_time_batch = time.time()

for idx, (_, row) in enumerate(paired_trials_df.iterrows(), start=1):
    manifest_rec, merged_df, failure_rec = merge_trial(row)

    if failure_rec is not None:
        failure_records.append(failure_rec)
        failed_manifest = {
            "trial_key": str(row["trial_key"]),
            "subject_id": str(row["subject_id"]),
            "condition": str(row["condition"]),
            "trial_id": row["trial_id"],
            "linear_file": str(row["linear_path"]),
            "angular_file": str(row["angular_path"]),
            "pairing_status": row["pairing_status"],
            "merge_status": "FAILED",
            "failure_reason": failure_rec.get("failure_reason", ""),
            "alignment_status": "FAILED",
            "alignment_method": "none",
            "linear_original_length": 0,
            "angular_original_length": 0,
            "merged_length": 0,
            "linear_start_time": np.nan,
            "linear_end_time": np.nan,
            "angular_start_time": np.nan,
            "angular_end_time": np.nan,
            "merged_start_time": np.nan,
            "merged_end_time": np.nan,
            "linear_sampling_interval": np.nan,
            "angular_sampling_interval": np.nan,
            "discarded_linear_samples": 0,
            "discarded_angular_samples": 0,
            "output_path": "",
            "validation_status": "FAILED"
        }
        manifest_records.append(failed_manifest)
    else:
        out_path = MERGED_DIR / f"{manifest_rec['trial_key']}.csv"
                
        merged_df.to_csv(out_path, index=False, float_format="%.17g")
        
        manifest_rec["merged_path"] = str(out_path.relative_to(PROJECT_ROOT))
        manifest_rec["output_path"] = str(out_path.relative_to(PROJECT_ROOT))
        manifest_records.append(manifest_rec)

    if idx % 100 == 0 or idx == len(paired_trials_df):
        elapsed = time.time() - start_time_batch
        print(f"Processed {idx}/{len(paired_trials_df)} ({elapsed:.1f}s)")

merged_manifest_df = pd.DataFrame(manifest_records)
merge_failures_df = pd.DataFrame(failure_records)

MANIFEST_PATH = METADATA_DIR / "merged_trial_manifest.csv"
FAILURES_PATH = TABLES_DIR / "merge_failures.csv"

merged_manifest_df.to_csv(MANIFEST_PATH, index=False)

# Ensure standardized failure columns
expected_failure_cols = [
    "trial_key", "subject_id", "condition", "trial_id",
    "failure_stage", "failure_type", "description", "linear_file", "angular_file"
]
if merge_failures_df.empty:
    merge_failures_df = pd.DataFrame(columns=expected_failure_cols)
else:
    if "failure_reason" in merge_failures_df.columns and "description" not in merge_failures_df.columns:
        merge_failures_df["description"] = merge_failures_df["failure_reason"]
    if "failure_type" not in merge_failures_df.columns:
        merge_failures_df["failure_type"] = "TEMPORAL_ALIGNMENT_FAILURE"
    for col in expected_failure_cols:
        if col not in merge_failures_df.columns:
            merge_failures_df[col] = ""
    merge_failures_df = merge_failures_df[expected_failure_cols]

merge_failures_df.to_csv(FAILURES_PATH, index=False)

successful_merges = int(merged_manifest_df["merge_status"].eq("SUCCESS").sum())
failed_merges = int(merged_manifest_df["merge_status"].eq("FAILED").sum())

print("=" * 70)
print("BATCH MERGE COMPLETE")
print("=" * 70)
print(f"Paired trials : {len(paired_trials_df)}")
print(f"Successful    : {successful_merges}")
print(f"Failed        : {failed_merges}")
print("=" * 70)



Processed 100/884 (25.3s)
Processed 200/884 (60.9s)
Processed 300/884 (99.7s)
Processed 400/884 (128.3s)
Processed 500/884 (156.3s)
Processed 600/884 (192.0s)
Processed 700/884 (217.7s)
Processed 800/884 (243.1s)
Processed 884/884 (262.8s)
BATCH MERGE COMPLETE
Paired trials : 884
Successful    : 881
Failed        : 3


## 7. Export Merged Trial Manifest & Failure Tables

We write the complete trial manifest to `data/metadata/merged_trial_manifest.csv` and the failure registry to `results/tables/merge_failures.csv`.


In [9]:
# ============================================================================
# 7. MANIFEST / OUTPUT / FAILURE RECONCILIATION
# ============================================================================

manifest_keys = set(merged_manifest_df["trial_key"].astype(str))
success_keys = set(
    merged_manifest_df.loc[
        merged_manifest_df["merge_status"].eq("SUCCESS"), "trial_key"
    ].astype(str)
)
failure_keys = set(
    merge_failures_df["trial_key"].dropna().astype(str)
)

if success_keys & failure_keys:
    raise RuntimeError("A trial is simultaneously SUCCESS and FAILED.")

paired_keys = set(paired_trials_df["trial_key"].astype(str))
if success_keys | failure_keys != paired_keys:
    raise RuntimeError("Every paired trial must be either SUCCESS or FAILED.")

actual_output_keys = {
    p.stem
    for p in MERGED_DIR.glob("*.csv")
}

if success_keys != actual_output_keys:
    raise RuntimeError(
        f"Actual merged outputs ({len(actual_output_keys)}) do not match manifest SUCCESS trials ({len(success_keys)})."
    )

if failure_keys & actual_output_keys:
    raise RuntimeError("A FAILED trial still has a merged output.")

print("=" * 70)
print("MANIFEST / OUTPUT RECONCILIATION")
print("=" * 70)
print("PASS: One status per paired trial.")
print("PASS: Merged outputs exactly match SUCCESS trials.")
print("PASS: FAILED trials have no merged output.")
print("PASS: Failure report exactly matches FAILED trials.")
print("=" * 70)



MANIFEST / OUTPUT RECONCILIATION
PASS: One status per paired trial.
PASS: Merged outputs exactly match SUCCESS trials.
PASS: FAILED trials have no merged output.
PASS: Failure report exactly matches FAILED trials.


## 8. Schema Consistency Across All Merged Trials

To confirm dataset uniformity for downstream modeling, we verify that every merged trial shares the exact same column schema and channel count, recording our findings in `results/tables/merge_schema_consistency.csv`.


In [10]:
# ============================================================================
# 8. FULL ORDERED SCHEMA VALIDATION
# ============================================================================

schema_records = []
expected_keys = set(merged_manifest_df[merged_manifest_df["merge_status"] == "SUCCESS"]["trial_key"].astype(str))

for p in sorted(MERGED_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(p, nrows=1)
        cols = df.columns.tolist()
        row_count = 1 # valid file
        is_readable = True
    except Exception:
        cols = []
        row_count = 0
        is_readable = False
        
    trial_key = p.stem
    
    # Check conditions
    has_frame = "frame" in cols
    has_time = "time" in cols
    has_lin = any(c.startswith("lin__") for c in cols)
    has_ang = any(c.startswith("ang__") for c in cols)
    no_unexpected = all(c in merged_columns for c in cols)
    no_duplicates = len(cols) == len(set(cols))
    schema_match = (cols == merged_columns)
    valid_row_count = row_count > 0
    valid_key = trial_key in expected_keys

    schema_records.append({
        "file": str(p.relative_to(PROJECT_ROOT)),
        "trial_key": trial_key,
        "is_readable": is_readable,
        "has_frame": has_frame,
        "has_time": has_time,
        "has_lin": has_lin,
        "has_ang": has_ang,
        "no_unexpected": no_unexpected,
        "no_duplicates": no_duplicates,
        "valid_row_count": valid_row_count,
        "valid_key": valid_key,
        "schema_match": schema_match and is_readable and valid_row_count and valid_key,
        "column_count": len(cols),
        "expected_column_count": len(merged_columns),
        "missing_columns": ";".join(c for c in merged_columns if c not in cols),
        "unexpected_columns": ";".join(c for c in cols if c not in merged_columns),
    })

schema_validation_df = pd.DataFrame(schema_records)
SCHEMA_VALIDATION_PATH = TABLES_DIR / "merge_schema_validation.csv"
schema_validation_df.to_csv(TABLES_DIR / "merged_schema_validation.csv", index=False)
schema_validation_df.to_csv(SCHEMA_VALIDATION_PATH, index=False)

assert len(schema_validation_df) == successful_merges, f"Schema validation checked {len(schema_validation_df)} files but {successful_merges} were successfully merged. Validator glob pattern might be wrong."

if not schema_validation_df.empty and not schema_validation_df["schema_match"].all():
    raise RuntimeError(
        f"Full ordered-schema validation failed. See {SCHEMA_VALIDATION_PATH}"
    )

schema_consistency_df = pd.DataFrame([{
    "successful_files_checked": len(schema_validation_df),
    "expected_columns": len(merged_columns),
    "schema_consistent": True,
}])
SCHEMA_CONSISTENCY_PATH = TABLES_DIR / "merge_schema_consistency.csv"
schema_consistency_df.to_csv(SCHEMA_CONSISTENCY_PATH, index=False)

print("=" * 70)
print("FULL ORDERED SCHEMA VALIDATION")
print("=" * 70)
print(f"Files checked : {len(schema_validation_df)}")
print(f"Columns/file  : {len(merged_columns)}")
print("PASS: Every successful merged file matches the canonical schema and has valid rows.")
print("=" * 70)


FULL ORDERED SCHEMA VALIDATION
Files checked : 881
Columns/file  : 182
PASS: Every successful merged file matches the canonical schema and has valid rows.


## 9. Merge Validation Summary

We aggregate overall merging statistics, breakdown of alignment classes, and discarded sample metrics into `results/tables/merge_validation_summary.csv`.


In [11]:
# ============================================================================
# 9. MERGE VALIDATION AND QUALITY SUMMARY
# ============================================================================

alignment_counts = (
    merged_manifest_df["alignment_status"]
    .value_counts(dropna=False)
    .to_dict()
)

successful_manifest = merged_manifest_df[
    merged_manifest_df["merge_status"].eq("SUCCESS")
]

reasons = merged_manifest_df.loc[merged_manifest_df["merge_status"] == "FAILED", "failure_reason"].fillna("")
schema_failures = int(reasons.str.contains("SCHEMA_MISMATCH").sum())
missing_time = int(reasons.str.contains("MISSING_TIME_INFORMATION").sum())
non_monotonic_time = int(reasons.str.contains("NON_MONOTONIC_TIME").sum())
non_monotonic_frame = int(reasons.str.contains("NON_MONOTONIC_FRAME").sum())
non_uniform_grid = int(reasons.str.contains("NON_UNIFORM_GRID").sum())
sampling_interval = int(reasons.str.contains("SAMPLING_INTERVAL_MISMATCH").sum())
no_common_samples = int(reasons.str.contains("NO_COMMON_SAMPLES").sum())
incompatible_timebase = int(reasons.str.contains("INCOMPATIBLE_TIMEBASE").sum())

known_failures = (schema_failures + missing_time + non_monotonic_time + non_monotonic_frame + 
                  non_uniform_grid + sampling_interval + no_common_samples + incompatible_timebase)
other_failures = int(len(reasons) - known_failures)

# 1. merge_validation_summary.csv
validation_summary_df = pd.DataFrame([{
    "total_paired_trials": len(paired_trials_df),
    "successfully_merged": successful_merges,
    "failed_merges": failed_merges,
    "exactly_aligned_trials": alignment_counts.get("EXACTLY_ALIGNED", 0),
    "same_grid_different_length_trials": alignment_counts.get("SAME_GRID_DIFFERENT_LENGTH", 0),
    "total_discarded_linear_samples": int(successful_manifest["discarded_linear_samples"].fillna(0).sum()) if not successful_manifest.empty else 0,
    "total_discarded_angular_samples": int(successful_manifest["discarded_angular_samples"].fillna(0).sum()) if not successful_manifest.empty else 0,
    "min_merged_length": int(successful_manifest["merged_length"].min()) if not successful_manifest.empty else np.nan,
    "max_merged_length": int(successful_manifest["merged_length"].max()) if not successful_manifest.empty else np.nan,
    "mean_merged_length": float(successful_manifest["merged_length"].mean()) if not successful_manifest.empty else np.nan,
}])

VALIDATION_SUMMARY_PATH = TABLES_DIR / "merge_validation_summary.csv"
validation_summary_df.to_csv(VALIDATION_SUMMARY_PATH, index=False)

# 2. merge_quality_summary.csv
total_trials_considered = len(nb02_index_df) if "nb02_index_df" in dir() else len(paired_trials_df)
quality_summary_df = pd.DataFrame([{
    "trials_considered": total_trials_considered,
    "paired_trials": len(paired_trials_df),
    "successful_merges": successful_merges,
    "failed_merges": failed_merges,
    "linear_only_trials": int((nb02_index_df["pairing_status"] == "LINEAR_ONLY").sum()) if "nb02_index_df" in dir() else 0,
    "angular_only_trials": int((nb02_index_df["pairing_status"] == "ANGULAR_ONLY").sum()) if "nb02_index_df" in dir() else 0,
    "duplicate_trials": int(nb02_index_df["pairing_status"].isin(["DUPLICATE_LINEAR", "DUPLICATE_ANGULAR"]).sum()) if "nb02_index_df" in dir() else 0,
    "ambiguous_trials": int((nb02_index_df["pairing_status"] == "AMBIGUOUS").sum()) if "nb02_index_df" in dir() else 0,
    "exactly_aligned_trials": alignment_counts.get("EXACTLY_ALIGNED", 0),
    "same_grid_different_length_trials": alignment_counts.get("SAME_GRID_DIFFERENT_LENGTH", 0),
    "schema_failures": schema_failures,
    "missing_time_failures": missing_time,
    "non_monotonic_time_failures": non_monotonic_time,
    "non_monotonic_frame_failures": non_monotonic_frame,
    "non_uniform_grid_failures": non_uniform_grid,
    "sampling_interval_failures": sampling_interval,
    "no_common_samples_failures": no_common_samples,
    "incompatible_timebase_failures": incompatible_timebase,
    "other_failures": other_failures,
}])

QUALITY_SUMMARY_PATH = TABLES_DIR / "merge_quality_summary.csv"
quality_summary_df.to_csv(QUALITY_SUMMARY_PATH, index=False)

print("=" * 70)
print("MERGE QUALITY & VALIDATION SUMMARY")
print("=" * 70)
for k, v in quality_summary_df.iloc[0].items():
    print(f"{k:<35}: {v}")
print("=" * 70)


MERGE QUALITY & VALIDATION SUMMARY
trials_considered                  : 884
paired_trials                      : 884
successful_merges                  : 881
failed_merges                      : 3
linear_only_trials                 : 0
angular_only_trials                : 0
duplicate_trials                   : 0
ambiguous_trials                   : 0
exactly_aligned_trials             : 878
same_grid_different_length_trials  : 3
schema_failures                    : 0
missing_time_failures              : 0
non_monotonic_time_failures        : 3
non_monotonic_frame_failures       : 0
non_uniform_grid_failures          : 0
sampling_interval_failures         : 0
no_common_samples_failures         : 0
incompatible_timebase_failures     : 0
other_failures                     : 0


## 10. Signal Value Integrity Verification

To formally guarantee that raw signal values were not modified, normalized, smoothed, or scaled, we perform an automated numerical verification comparing raw source arrays against generated merged CSV values on a deterministic sample of trials and channels.


In [12]:
# ============================================================================
# 10. FULL SIGNAL VALUE INTEGRITY AUDIT
# ============================================================================

def arrays_equal(a, b, atol=1e-10):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return (
        a.shape == b.shape
        and np.allclose(a, b, rtol=0.0, atol=atol, equal_nan=True)
    )

value_integrity_records = []

for _, rec in successful_manifest.iterrows():

    row = paired_trials_df[
        paired_trials_df["trial_key"].astype(str)
        == str(rec["trial_key"])
    ].iloc[0]

    lcols, _, ldf = read_kinematics_xls(row["linear_full_path"])
    acols, _, adf = read_kinematics_xls(row["angular_full_path"])

    ltc, lfc = find_column(lcols, "Time"), find_column(lcols, "Frame")
    atc, afc = find_column(acols, "Time"), find_column(acols, "Frame")

    lt, lf = validate_time_frame(ldf, ltc, lfc)
    at, af = validate_time_frame(adf, atc, afc)

    pairs = existing_sample_matches(lt, lf, at, af)

    li = np.array([p[0] for p in pairs], dtype=int)
    ai = np.array([p[1] for p in pairs], dtype=int)

    merged = pd.read_csv(PROJECT_ROOT / rec["output_path"])

    checks = [
        arrays_equal(merged["frame"].to_numpy(), lf[li]),
        arrays_equal(merged["time"].to_numpy(), lt[li], atol=TIME_ABS_TOL),
    ]

    for c in canonical_linear_cols:
        checks.append(
            arrays_equal(
                merged[f"lin__{c}"].to_numpy(),
                ldf[c].to_numpy()[li]
            )
        )

    for c in canonical_angular_cols:
        checks.append(
            arrays_equal(
                merged[f"ang__{c}"].to_numpy(),
                adf[c].to_numpy()[ai]
            )
        )

    value_integrity_records.append({
        "trial_key": rec["trial_key"],
        "all_values_match_raw_source": bool(all(checks)),
        "checks_per_trial": len(checks),
    })

value_integrity_df = pd.DataFrame(value_integrity_records)
VALUE_INTEGRITY_PATH = TABLES_DIR / "merge_value_integrity.csv"
value_integrity_df.to_csv(VALUE_INTEGRITY_PATH, index=False)

if not value_integrity_df.empty and not value_integrity_df["all_values_match_raw_source"].all():
    bad = value_integrity_df.loc[
        ~value_integrity_df["all_values_match_raw_source"], "trial_key"
    ].tolist()
    raise RuntimeError(f"Raw-value integrity failed: {bad[:10]}")

print("=" * 70)
print("FULL SIGNAL VALUE INTEGRITY AUDIT")
print("=" * 70)
print(f"Successful trials checked : {len(value_integrity_df)}")
print("PASS: Every retained signal value matches its raw source.")
print("PASS: Frame/time coordinates match their validated source grid.")
print("=" * 70)

FULL SIGNAL VALUE INTEGRITY AUDIT
Successful trials checked : 881
PASS: Every retained signal value matches its raw source.
PASS: Frame/time coordinates match their validated source grid.


## 11. Alignment Diagnostic Visualizations

We generate a focused diagnostic figure showing temporal coverage and alignment properties for representative valid trials:
1. **Direct Sample-by-Sample Synchronized Trial (`EXACTLY_ALIGNED`)**
2. **Common Temporal Interval Trial (`SAME_GRID_DIFFERENT_LENGTH`)**

The figure illustrates temporal duration, endpoint alignment, and discarded sample boundaries.


In [13]:
# ============================================================================
# 11. ALIGNMENT DIAGNOSTIC VISUALIZATION — ROBUST
# ============================================================================

from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ---------------------------------------------------------------------------
# Validate manifest availability
# ---------------------------------------------------------------------------

if "merged_manifest_df" not in globals():
    raise RuntimeError(
        "merged_manifest_df is not available. "
        "Run the batch merge cell before generating diagnostics."
    )

if merged_manifest_df.empty:
    raise RuntimeError(
        "The merged trial manifest is empty. "
        "There are no trials available for alignment diagnostics."
    )


# ---------------------------------------------------------------------------
# Select available successful alignment classes
# ---------------------------------------------------------------------------

successful_manifest = merged_manifest_df[
    merged_manifest_df["merge_status"].eq("SUCCESS")
].copy()

if successful_manifest.empty:
    print("=" * 70)
    print("ALIGNMENT DIAGNOSTIC VISUALIZATION")
    print("=" * 70)
    print("No successful merged trials are available.")
    print(
        "Diagnostic visualization skipped because all paired trials "
        "were unsuccessful."
    )
    print("=" * 70)

else:

    exact_trials = successful_manifest[
        successful_manifest["alignment_status"].eq("EXACTLY_ALIGNED")
    ]

    same_grid_trials = successful_manifest[
        successful_manifest["alignment_status"].eq(
            "SAME_GRID_DIFFERENT_LENGTH"
        )
    ]


    # -----------------------------------------------------------------------
    # Build a list of available diagnostic cases
    # -----------------------------------------------------------------------

    diagnostic_cases = []

    if not exact_trials.empty:
        diagnostic_cases.append(
            (
                "EXACTLY_ALIGNED",
                exact_trials.iloc[0]
            )
        )

    if not same_grid_trials.empty:
        diagnostic_cases.append(
            (
                "SAME_GRID_DIFFERENT_LENGTH",
                same_grid_trials.iloc[0]
            )
        )


    # -----------------------------------------------------------------------
    # No recognized successful alignment classes
    # -----------------------------------------------------------------------

    if not diagnostic_cases:

        print("=" * 70)
        print("ALIGNMENT DIAGNOSTIC VISUALIZATION")
        print("=" * 70)

        print(
            "No recognized successful alignment classes are available."
        )

        print(
            "Observed successful alignment statuses:"
        )

        print(
            successful_manifest["alignment_status"]
            .value_counts(dropna=False)
        )

        print("=" * 70)

    else:

        # ---------------------------------------------------------------
        # One axis per available alignment case
        # ---------------------------------------------------------------

        n_cases = len(diagnostic_cases)

        fig, axes = plt.subplots(
            n_cases,
            1,
            figsize=(12, 4.5 * n_cases),
            squeeze=False
        )

        axes = axes.ravel()


        for ax, (alignment_status, trial_row) in zip(
            axes,
            diagnostic_cases
        ):

            trial_key = trial_row["trial_key"]

            lin_start = float(
                trial_row["linear_start_time"]
            )

            lin_end = float(
                trial_row["linear_end_time"]
            )

            ang_start = float(
                trial_row["angular_start_time"]
            )

            ang_end = float(
                trial_row["angular_end_time"]
            )

            merged_start = float(
                trial_row["merged_start_time"]
            )

            merged_end = float(
                trial_row["merged_end_time"]
            )


            # -----------------------------------------------------------
            # Durations
            # -----------------------------------------------------------

            linear_duration = lin_end - lin_start
            angular_duration = ang_end - ang_start
            merged_duration = merged_end - merged_start


            # -----------------------------------------------------------
            # Horizontal temporal coverage
            # -----------------------------------------------------------

            labels = [
                "Merged Stream",
                "Angular Kinematics",
                "Linear Kinematics"
            ]

            durations = [
                merged_duration,
                angular_duration,
                linear_duration
            ]

            starts = [
                merged_start,
                ang_start,
                lin_start
            ]


            ax.barh(
                labels,
                durations,
                left=starts,
                height=0.45,
                alpha=0.85
            )


            # -----------------------------------------------------------
            # Title
            # -----------------------------------------------------------

            if alignment_status == "EXACTLY_ALIGNED":

                title = (
                    "Case 1: EXACTLY_ALIGNED — "
                    f"Trial: {trial_key} "
                    f"(Lin={int(trial_row['linear_original_length'])}, "
                    f"Ang={int(trial_row['angular_original_length'])})"
                )

            elif alignment_status == "SAME_GRID_DIFFERENT_LENGTH":

                title = (
                    "Case 2: SAME_GRID_DIFFERENT_LENGTH — "
                    f"Trial: {trial_key} "
                    f"(Discarded Lin="
                    f"{int(trial_row['discarded_linear_samples'])}, "
                    f"Ang="
                    f"{int(trial_row['discarded_angular_samples'])})"
                )

                # Mark the actual merged interval boundaries.
                ax.axvline(
                    merged_start,
                    linestyle=":",
                    linewidth=1.5,
                    label="Merged interval start"
                )

                ax.axvline(
                    merged_end,
                    linestyle=":",
                    linewidth=1.5,
                    label="Merged interval end"
                )

                ax.legend(
                    loc="lower right"
                )

            else:

                title = (
                    f"{alignment_status} — "
                    f"Trial: {trial_key}"
                )


            ax.set_title(
                title,
                fontsize=12,
                fontweight="bold"
            )

            ax.set_xlabel(
                "Time (seconds)",
                fontsize=11
            )

            ax.grid(
                axis="x",
                linestyle="--",
                alpha=0.6
            )


        # -------------------------------------------------------------------
        # Save diagnostic figure
        # -------------------------------------------------------------------

        plt.tight_layout()

        DIAGNOSTIC_PLOT_PATH = (
            Path(FIGURES_DIR)
            / "merge_alignment_diagnostics.png"
        )

        DIAGNOSTIC_PLOT_PATH.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        plt.savefig(
            DIAGNOSTIC_PLOT_PATH,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

        print("=" * 70)
        print("ALIGNMENT DIAGNOSTIC VISUALIZATION")
        print("=" * 70)

        print(
            f"Successful trials available : "
            f"{len(successful_manifest)}"
        )

        print(
            f"EXACTLY_ALIGNED trials       : "
            f"{len(exact_trials)}"
        )

        print(
            f"SAME_GRID_DIFFERENT_LENGTH  : "
            f"{len(same_grid_trials)}"
        )

        print(
            "Diagnostic cases plotted     : "
            f"{len(diagnostic_cases)}"
        )

        print(
            "Saved to                     : "
            f"{DIAGNOSTIC_PLOT_PATH.relative_to(PROJECT_ROOT)}"
        )

        print("=" * 70)

ALIGNMENT DIAGNOSTIC VISUALIZATION
Successful trials available : 881
EXACTLY_ALIGNED trials       : 878
SAME_GRID_DIFFERENT_LENGTH  : 3
Diagnostic cases plotted     : 2
Saved to                     : figures\preprocessing\merge_alignment_diagnostics.png


C:\Users\lab\AppData\Local\Temp\ipykernel_26152\3676128253.py:291: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Post-Execution Raw Data Integrity Verification

We re-scan `data/raw/` to ensure that no raw file was created, renamed, modified, moved, or deleted during merging.


In [14]:
# ============================================================================
# 12. POST-EXECUTION RAW DATA INTEGRITY
# ============================================================================

raw_files_post = []

for root, _, files in os.walk(RAW_DIR):
    for name in sorted(files):
        p = Path(root) / name
        raw_files_post.append({
            "relative_path": str(p.relative_to(RAW_DIR)),
            "size_bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

df_raw_post = (
    pd.DataFrame(raw_files_post)
    .sort_values("relative_path")
    .reset_index(drop=True)
)

RAW_INTEGRITY_POST_PATH = TABLES_DIR / "raw_integrity_post.csv"
df_raw_post.to_csv(RAW_INTEGRITY_POST_PATH, index=False)

raw_integrity_pass = df_raw_pre.equals(df_raw_post)

print("=" * 70)
print("POST-EXECUTION RAW DATA INTEGRITY")
print("=" * 70)
print(f"Files before : {len(df_raw_pre)}")
print(f"Files after  : {len(df_raw_post)}")
print(f"SHA-256/path/size unchanged : {raw_integrity_pass}")
print("=" * 70)

if not raw_integrity_pass:
    raise RuntimeError(
        "Raw data changed during Notebook 03."
    )

POST-EXECUTION RAW DATA INTEGRITY
Files before : 4831
Files after  : 4831
SHA-256/path/size unchanged : True


## 13. Final Programmatic Merging Audit & Downstream Handoff Gate

We evaluate all 15 audit requirements programmatically to determine readiness for Notebook 04.


In [15]:
# ============================================================================
# 13. FINAL PROGRAMMATIC AUDIT + NOTEBOOK 04 HANDOFF GATE
# ============================================================================

required_outputs = [
    MANIFEST_PATH,
    FAILURES_PATH,
    PROVENANCE_PATH,
    VALIDATION_SUMMARY_PATH,
    QUALITY_SUMMARY_PATH,
    SCHEMA_VALIDATION_PATH,
    SCHEMA_CONSISTENCY_PATH,
    VALUE_INTEGRITY_PATH,
    RAW_INTEGRITY_PRE_PATH,
    RAW_INTEGRITY_POST_PATH,
]

outputs_exist = all(Path(p).exists() for p in required_outputs)

audit_criteria = {
    "Notebook 02 handoff loaded": len(paired_trials_df) > 0,
    "All paired files exist": bool(
        paired_trials_df["linear_exists"].all()
        and paired_trials_df["angular_exists"].all()
    ),
    "One status per paired trial": (successful_merges + failed_merges) == len(paired_trials_df),
    "SUCCESS outputs reconciled": actual_output_keys == success_keys,
    "FAILED trials have no outputs": not bool(failure_keys & actual_output_keys),
    "Failure report reconciled": len(merge_failures_df) == failed_merges,
    "Full ordered schema passed": (
        schema_validation_df.empty
        or bool(schema_validation_df["schema_match"].all())
    ),
    "Feature provenance matches schema": (
        provenance_df["merged_feature_name"].tolist() == merged_columns
    ),
    "Full signal value integrity passed": (
        value_integrity_df.empty
        or bool(value_integrity_df["all_values_match_raw_source"].all())
    ),
    "Raw data unchanged": bool(raw_integrity_pass),
    "Required artifacts exist": bool(outputs_exist),
}

audit_df = pd.DataFrame([
    {"criterion": k, "passed": bool(v)}
    for k, v in audit_criteria.items()
])

AUDIT_PATH = TABLES_DIR / "merge_final_audit.csv"
audit_df.to_csv(AUDIT_PATH, index=False)

all_passed = bool(audit_df["passed"].all())

print("=" * 70)
print("NOTEBOOK 03 — FINAL AUDIT")
print("=" * 70)

for _, r in audit_df.iterrows():
    print(f"{'PASS' if r['passed'] else 'FAIL':<6} {r['criterion']}")

print("-" * 70)
print(f"Paired trials        : {len(paired_trials_df)}")
print(f"Successful merges    : {successful_merges}")
print(f"Documented failures : {failed_merges}")
print(f"Audit report        : {AUDIT_PATH.relative_to(PROJECT_ROOT)}")

if not all_passed:
    raise RuntimeError("Notebook 03 is NOT READY. Resolve failed audit criteria first.")

if failed_merges:
    final_status = "READY_WITH_DOCUMENTED_MERGE_FAILURES"
else:
    final_status = "READY"

print(f"FINAL STATUS: {final_status}")
print("=" * 70)
print()
print("Notebook 04 handoff:")
print(f"  Merged data : {MERGED_DIR.relative_to(PROJECT_ROOT)}")
print(f"  Manifest    : {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"  Provenance  : {PROVENANCE_PATH.relative_to(PROJECT_ROOT)}")
print("  Use only merge_status == SUCCESS trials for downstream signal analysis.")



NOTEBOOK 03 — FINAL AUDIT
PASS   Notebook 02 handoff loaded
PASS   All paired files exist
PASS   One status per paired trial
PASS   SUCCESS outputs reconciled
PASS   FAILED trials have no outputs
PASS   Failure report reconciled
PASS   Full ordered schema passed
PASS   Feature provenance matches schema
PASS   Full signal value integrity passed
PASS   Raw data unchanged
PASS   Required artifacts exist
----------------------------------------------------------------------
Paired trials        : 884
Successful merges    : 881
Documented failures : 3
Audit report        : results\tables\merge_final_audit.csv
FINAL STATUS: READY_WITH_DOCUMENTED_MERGE_FAILURES

Notebook 04 handoff:
  Merged data : data\merged
  Manifest    : data\metadata\merged_trial_manifest.csv
  Provenance  : data\metadata\merged_feature_provenance.csv
  Use only merge_status == SUCCESS trials for downstream signal analysis.


## Final Research-Grade Audit

Before Notebook 03 is considered complete, confirm programmatically that:

- Every paired trial has a deterministic trial key.
- Exact alignment requires matching `Frame` and `Time` sample-by-sample.
- Time is strictly increasing; duplicate timestamps are rejected.
- Same-grid/different-length trials retain only their common temporal interval and explicitly record discarded samples and endpoints.
- Incompatible timebases are rejected rather than silently resampled.
- Merged columns use `lin__` and `ang__` prefixes, with validated shared `frame`/`time` coordinates.
- The complete ordered merged schema is identical across every successful merged trial.
- Feature provenance is consistent with that validated schema.
- Raw signal values are not transformed by this notebook.
- Raw input file count/size integrity checks pass after execution.
- Failures are written to `results/tables/merge_failures.csv` and successful trials to the merged manifest.

**No downstream preprocessing is performed here:** no filtering, interpolation, resampling, normalization, imputation, smoothing, detrending, clipping, graph construction, GSN/TBSN, scaling, windowing, or modeling.

**Execution requirement:** Run the notebook from a fresh kernel using **Restart & Run All**. No validation step may depend on variables created only by prior interactive execution.


In [16]:
# ============================================================================
# FINAL TRIAL COMPLETENESS AUDIT
# ============================================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
MERGED_DIR = PROJECT_ROOT / "data" / "merged"
TABLES_DIR = PROJECT_ROOT / "results" / "tables"

manifest_path = METADATA_DIR / "merged_trial_manifest.csv"
failures_path = TABLES_DIR / "merge_failures.csv"

if not manifest_path.exists():
    raise FileNotFoundError(f"Required manifest not found: {manifest_path}")

if not failures_path.exists():
    raise FileNotFoundError(f"Required failure report not found: {failures_path}")

manifest_df = pd.read_csv(manifest_path)
failures_df = pd.read_csv(failures_path)

required_manifest_cols = {"trial_key", "merge_status"}
missing_manifest_cols = required_manifest_cols - set(manifest_df.columns)

if missing_manifest_cols:
    raise RuntimeError(f"Manifest is missing required columns: {sorted(missing_manifest_cols)}")

manifest_keys = set(manifest_df["trial_key"].dropna().astype(str).str.strip())
manifest_success_keys = set(manifest_df.loc[manifest_df["merge_status"].eq("SUCCESS"), "trial_key"].dropna().astype(str).str.strip())
manifest_failed_keys = set(manifest_df.loc[manifest_df["merge_status"].eq("FAILED"), "trial_key"].dropna().astype(str).str.strip())

# Actual merged outputs
successful_merge_files = sorted(MERGED_DIR.glob("*.csv"))
file_success_keys = {f.stem.removesuffix("_merged") for f in successful_merge_files}

# Failure report
failure_key_col = next((c for c in ["trial_key", "pair_key", "trial_id"] if c in failures_df.columns), None)
if failure_key_col is None:
    raise RuntimeError("Failure report does not contain a usable trial-key column.")
failure_report_keys = set(failures_df[failure_key_col].dropna().astype(str).str.strip())

# Calculate failure reasons
reasons = manifest_df.loc[manifest_df["merge_status"] == "FAILED", "failure_reason"].fillna("")
schema_failures = int(reasons.str.contains("SCHEMA_MISMATCH").sum())
missing_time = int(reasons.str.contains("MISSING_TIME_INFORMATION").sum())
non_monotonic_time = int(reasons.str.contains("NON_MONOTONIC_TIME").sum())
non_monotonic_frame = int(reasons.str.contains("NON_MONOTONIC_FRAME").sum())
non_uniform_grid = int(reasons.str.contains("NON_UNIFORM_GRID").sum())
sampling_interval = int(reasons.str.contains("SAMPLING_INTERVAL_MISMATCH").sum())
no_common_samples = int(reasons.str.contains("NO_COMMON_SAMPLES").sum())
incompatible_timebase = int(reasons.str.contains("INCOMPATIBLE_TIMEBASE").sum())

known_failures = (schema_failures + missing_time + non_monotonic_time + non_monotonic_frame + 
                  non_uniform_grid + sampling_interval + no_common_samples + incompatible_timebase)
other_failures = int(len(reasons) - known_failures)

try:
    schema_val_df = pd.read_csv(TABLES_DIR / "merge_schema_validation.csv")
    files_validated = len(schema_val_df)
    files_passing = schema_val_df["schema_match"].sum()
except Exception:
    files_validated = 0
    files_passing = 0

print("=" * 70)
print("FINAL TRIAL COMPLETENESS AUDIT")
print("=" * 70)
print(f"Candidate paired trials      : {len(manifest_keys)}")
print(f"Successful merges            : {len(manifest_success_keys)}")
print(f"Failed merges                : {len(manifest_failed_keys)}")
print(f"  Schema failures            : {schema_failures}")
print(f"  Missing-time failures      : {missing_time}")
print(f"  Non-monotonic-time failures: {non_monotonic_time}")
print(f"  Non-monotonic-frame failur.: {non_monotonic_frame}")
print(f"  Non-uniform-grid failures  : {non_uniform_grid}")
print(f"  Sampling-interval failures : {sampling_interval}")
print(f"  No-common-sample failures  : {no_common_samples}")
print(f"  Incompatible-timebase      : {incompatible_timebase}")
print(f"  Other failures             : {other_failures}")
print(f"Merged files written         : {len(file_success_keys)}")
print(f"Merged files validated       : {files_validated}")
print(f"Merged files passing val.    : {files_passing}")

# Checks
status_union = manifest_success_keys | manifest_failed_keys
status_overlap = manifest_success_keys & manifest_failed_keys

if status_overlap:
    raise RuntimeError(f"FAIL: Trials are simultaneously SUCCESS and FAILED: {sorted(status_overlap)[:10]}")

if status_union != manifest_keys:
    unclassified = manifest_keys - status_union
    raise RuntimeError(f"FAIL: Manifest contains unclassified trials: {sorted(unclassified)[:10]}")

missing_success_outputs = manifest_success_keys - file_success_keys
unexpected_outputs = file_success_keys - manifest_success_keys
failed_with_outputs = manifest_failed_keys & file_success_keys

if missing_success_outputs:
    raise RuntimeError(f"FAIL: SUCCESS trials without merged outputs: {sorted(missing_success_outputs)[:10]}")

if unexpected_outputs:
    raise RuntimeError(f"FAIL: Merged outputs not marked SUCCESS: {sorted(unexpected_outputs)[:10]}")

if failed_with_outputs:
    raise RuntimeError(f"FAIL: FAILED trials still have merged outputs: {sorted(failed_with_outputs)[:10]}")

if failure_report_keys != manifest_failed_keys:
    missing_failure_records = manifest_failed_keys - failure_report_keys
    unexpected_failure_records = failure_report_keys - manifest_failed_keys
    raise RuntimeError(f"FAIL: Failure report does not exactly match manifest failures.")

reconciled_count = len(manifest_success_keys) + len(manifest_failed_keys)
if reconciled_count != len(manifest_keys):
    raise RuntimeError("FAIL: SUCCESS + FAILED does not equal manifest trial count.")

if files_validated != len(manifest_success_keys):
    raise RuntimeError("FAIL: validated merged-file count != successful merge count.")

if files_passing != files_validated:
    raise RuntimeError("FAIL: not all validated files passed schema match.")

print("PASS: Manifest trial statuses form a complete partition.")
print("PASS: Actual merged outputs exactly match SUCCESS trials.")
print("PASS: FAILED trials have no merged output.")
print("PASS: Failure report exactly matches FAILED trials.")
print("PASS: Validated file count exactly matches SUCCESS trials.")
print(f"PASS: {reconciled_count} manifest trials are fully reconciled.")
print()
print("=" * 70)
if manifest_failed_keys:
    print("FINAL STATUS: PASS WITH DOCUMENTED MERGE FAILURES")
    print("Documented failures are excluded from merged outputs. Downstream notebooks must use successful merged trials only.")
else:
    print("FINAL STATUS: PASS")
print("=" * 70)


FINAL TRIAL COMPLETENESS AUDIT
Candidate paired trials      : 884
Successful merges            : 881
Failed merges                : 3
  Schema failures            : 0
  Missing-time failures      : 0
  Non-monotonic-time failures: 3
  Non-monotonic-frame failur.: 0
  Non-uniform-grid failures  : 0
  Sampling-interval failures : 0
  No-common-sample failures  : 0
  Incompatible-timebase      : 0
  Other failures             : 0
Merged files written         : 881
Merged files validated       : 881
Merged files passing val.    : 881
PASS: Manifest trial statuses form a complete partition.
PASS: Actual merged outputs exactly match SUCCESS trials.
PASS: FAILED trials have no merged output.
PASS: Failure report exactly matches FAILED trials.
PASS: Validated file count exactly matches SUCCESS trials.
PASS: 884 manifest trials are fully reconciled.

FINAL STATUS: PASS WITH DOCUMENTED MERGE FAILURES
Documented failures are excluded from merged outputs. Downstream notebooks must use successful m

## Final Execution Requirement

Run this notebook from a **fresh kernel** using **Restart & Run All**.

The notebook removes only stale generated `*_merged.csv` outputs before the
current merge execution. It does **not** modify `data/raw/`.

Do not proceed to Notebook 04 until:

1. the merge execution completes;
2. authoritative full ordered-schema validation passes;
3. raw-data integrity passes; and
4. the final trial completeness audit passes.

If genuine source-data merge failures remain, they must be explicitly present
in both the manifest and `merge_failures.csv`, with no corresponding merged
CSV output.


## Final execution requirement

Run this notebook using **Restart Kernel → Run All**.

Do not manually execute cells out of order.

Notebook 03 is complete only when the final audit reports `READY` or
`READY_WITH_DOCUMENTED_MERGE_FAILURES`.

Documented merge failures are retained in the manifest and failure report and
must have no corresponding merged CSV output. They are not silently repaired
in this notebook.